In [19]:
import pandas as pd
import glob

# načtení všech CSV souborů jednotlivých krajů ze složky
kraje = glob.glob(
    "C:/Dominika/industrial_analysis/data/zamestnanci_odvetvi/csv/*.csv"
)

# spojení do jednoho DataFrame pro celou Českou republiku
zamestnanci_komplet = pd.concat([pd.read_csv(f) for f in kraje], ignore_index=True)

# odstranění posledního sloupce bez hodnot a řádků ve sloupcích Kraj, Zaměstnáni v NH, Pohlaví s prázdnou hodnotou
zamestnanci_komplet = zamestnanci_komplet.drop(columns=["Unnamed: 2"], errors="ignore")
zamestnanci_komplet = zamestnanci_komplet.dropna(
    subset=["Kraj", "ZAMĚSTNANÍ V NH", "Pohlaví"]
).copy()

# převod roků do jednoho sloupce
zamestnanci_komplet = zamestnanci_komplet.melt(
    id_vars=["Kraj", "ZAMĚSTNANÍ V NH", "Pohlaví"],
    var_name="Rok",
    value_name="Pocet_zamestnancu",
)

# převod roků na int
zamestnanci_komplet["Rok"] = zamestnanci_komplet["Rok"].astype(int)

# převedení znaků ".", "-", NaN na 0 ve sloupcích s odvětvími činnosti a následně převod na int
zamestnanci_komplet["Pocet_zamestnancu"] = (
    zamestnanci_komplet["Pocet_zamestnancu"]
    .replace([".", "-"], 0)
    .replace(",", ".", regex=True)
    .pipe(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)
# ověření absence NaN hodnot v souboru
assert zamestnanci_komplet.isna().sum().sum() == 0
#sečtení mužů a žen
zamestnanci_komplet = (
    zamestnanci_komplet
    .groupby(["Kraj", "Rok", "ZAMĚSTNANÍ V NH"], as_index=False)["Pocet_zamestnancu"]
    .sum()
)

# přejmenování
zamestnanci_komplet = zamestnanci_komplet.rename(
    columns={"ZAMĚSTNANÍ V NH": "Odvetvi"})

# seřazení
zamestnanci_komplet = zamestnanci_komplet.sort_values(["Kraj", "Rok"])

zamestnanci_komplet.to_csv("zamestnanci_odvetvi.csv", index=False)
